# Pipeline Completo — PDF → Markdown → Chunking → Embeddings → JSON

Executa o pipeline completo da AULA 04:
1. Faz upload dos PDFs
2. Converte para Markdown com `pymupdf4llm`
3. Aplica 10 estratégias de chunking com LangChain
4. Gera embeddings via OpenRouter
5. Salva `chunks_embeddings.json` por documento/estratégia
6. Gera `summary.json`

---
### 1. Instalação

In [ ]:
!pip install -q openai langchain-text-splitters pymupdf4llm numpy pandas

---
### 2. Configuração da API e Constantes

In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pymupdf4llm
from google.colab import userdata, files
from openai import OpenAI
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_API_KEY')
)

MAX_CHARS       = 6000
EMBEDDING_MODEL = 'openai/text-embedding-3-small'
BATCH_SIZE      = 20
SLEEP_BETWEEN   = 1.5

BASE_DIR = Path('/content/results')
BASE_DIR.mkdir(exist_ok=True)

print('Configuração concluída!')

---
### 3. Funções Auxiliares

In [ ]:
def limpar_texto(texto):
    texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto)
    linhas = [l for l in texto.splitlines() if len(l.strip()) > 4 or l.strip() == '']
    return '\n'.join(linhas)

def split_por_sentencas_agrupadas(texto, n=3):
    sentencas = re.split(r'(?<=[.!?])\s+', texto.strip())
    sentencas = [s.strip() for s in sentencas if len(s.strip()) > 10]
    chunks = []
    for i in range(0, len(sentencas), n):
        grupo = sentencas[i:i + n]
        chunks.append(' '.join(grupo))
    return [c for c in chunks if len(c) > 10]

def gerar_embeddings(textos):
    vetores = []
    for i in range(0, len(textos), BATCH_SIZE):
        batch = [str(t)[:MAX_CHARS] for t in textos[i:i + BATCH_SIZE]]
        for tentativa in range(3):
            try:
                resp = client.embeddings.create(input=batch, model=EMBEDDING_MODEL)
                vetores.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                if tentativa < 2:
                    time.sleep(5)
                else:
                    vetores.extend([[0.0] * 1536] * len(batch))
        if i + BATCH_SIZE < len(textos):
            time.sleep(SLEEP_BETWEEN)
    return vetores

def estatisticas_chunks(chunks):
    tamanhos = [len(str(c)) for c in chunks]
    if not tamanhos:
        return {'total': 0, 'media': 0, 'min': 0, 'max': 0}
    return {'total': len(chunks), 'media': int(np.mean(tamanhos)), 'min': min(tamanhos), 'max': max(tamanhos)}

print('Funções auxiliares prontas!')

---
### 4. Definição das 10 Estratégias

In [ ]:
ESTRATEGIAS = [
    {'test_id': 1, 'tipo': 'texto',    'strategy': 'fixed_200_no_overlap',    'nome': 'Fixo 200, sem overlap',        'chunk_size': 200,  'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='', chunk_size=200, chunk_overlap=0)},
    {'test_id': 2, 'tipo': 'texto',    'strategy': 'fixed_500_no_overlap',    'nome': 'Fixo 500, sem overlap',        'chunk_size': 500,  'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='', chunk_size=500, chunk_overlap=0)},
    {'test_id': 3, 'tipo': 'texto',    'strategy': 'fixed_1000_no_overlap',   'nome': 'Fixo 1000, sem overlap',       'chunk_size': 1000, 'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='', chunk_size=1000, chunk_overlap=0)},
    {'test_id': 4, 'tipo': 'texto',    'strategy': 'fixed_2000_no_overlap',   'nome': 'Fixo 2000, sem overlap',       'chunk_size': 2000, 'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='', chunk_size=2000, chunk_overlap=0)},
    {'test_id': 5, 'tipo': 'texto',    'strategy': 'fixed_500_overlap_50',    'nome': 'Fixo 500, overlap 50',         'chunk_size': 500,  'chunk_overlap': 50,
     'splitter': CharacterTextSplitter(separator='', chunk_size=500, chunk_overlap=50)},
    {'test_id': 6, 'tipo': 'texto',    'strategy': 'fixed_500_overlap_200',   'nome': 'Fixo 500, overlap 200',        'chunk_size': 500,  'chunk_overlap': 200,
     'splitter': CharacterTextSplitter(separator='', chunk_size=500, chunk_overlap=200)},
    {'test_id': 7, 'tipo': 'paragrafo','strategy': 'by_paragraph',            'nome': 'Por parágrafo',                'chunk_size': 5000, 'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='\n\n', chunk_size=5000, chunk_overlap=0, is_separator_regex=False)},
    {'test_id': 8, 'tipo': 'sentenca', 'strategy': 'by_sentence_grouped_3',   'nome': 'Por sentença (3 agrupadas)',   'chunk_size': None, 'chunk_overlap': 0,
     'splitter': None},
    {'test_id': 9, 'tipo': 'texto',    'strategy': 'recursive_hierarchical',  'nome': 'Recursivo hierárquico',        'chunk_size': 500,  'chunk_overlap': 50,
     'splitter': RecursiveCharacterTextSplitter(separators=['\n\n', '\n', '. ', '! ', '? ', ' ', ''], chunk_size=500, chunk_overlap=50)},
    {'test_id': 10,'tipo': 'markdown', 'strategy': 'markdown_headers',        'nome': 'Markdown headers (H1/H2/H3)', 'chunk_size': None, 'chunk_overlap': 0,
     'splitter': MarkdownHeaderTextSplitter(headers_to_split_on=[('#', 'h1'), ('##', 'h2'), ('###', 'h3')])},
]

print(f'{len(ESTRATEGIAS)} estratégias definidas!')

---
### 5. Upload dos PDFs e Conversão para Markdown

In [ ]:
print('Faça upload dos arquivos PDF:')
uploaded = files.upload()

documentos = {}  # doc_id → {'nome': ..., 'texto': ...}

for nome_arquivo, conteudo in uploaded.items():
    print(f'Convertendo {nome_arquivo}...', end=' ')
    caminho_tmp = f'/tmp/{nome_arquivo}'
    with open(caminho_tmp, 'wb') as f:
        f.write(conteudo)
    try:
        md = pymupdf4llm.to_markdown(caminho_tmp)
        md = limpar_texto(md)
        titulo = nome_arquivo.replace('.pdf', '').replace('_', ' ').title()
        md = f'# {titulo}\n\n{md}'
        doc_id = re.sub(r'[^a-z0-9_]', '_', nome_arquivo.replace('.pdf', '').lower())

        # Salvar markdown
        pasta_md = BASE_DIR / doc_id / 'markdown'
        pasta_md.mkdir(parents=True, exist_ok=True)
        (pasta_md / f'{doc_id}.md').write_text(md, encoding='utf-8')

        documentos[doc_id] = {'nome': nome_arquivo, 'texto': md}
        print(f'OK ({len(md):,} chars)')
    except Exception as e:
        print(f'ERRO: {e}')

print(f'\nTotal: {len(documentos)} documentos prontos.')

---
### 6. Execução do Pipeline (Chunking + Embeddings + JSON)

In [ ]:
summary_docs = []

for doc_id, info in documentos.items():
    doc_nome = info['nome']
    texto = info['texto']
    print(f'\n{"="*60}')
    print(f'Documento: {doc_nome} ({len(texto):,} chars)')
    print('='*60)

    experimentos = []

    for est in ESTRATEGIAS:
        test_id = est['test_id']
        print(f'  Teste {test_id:02d}: {est["nome"]}')

        # Dividir
        try:
            if est['tipo'] == 'markdown':
                docs_split = est['splitter'].split_text(texto)
                chunk_items = [{'text': d.page_content, 'meta': dict(d.metadata)}
                               for d in docs_split if d.page_content.strip()]
            elif est['tipo'] == 'sentenca':
                txts = split_por_sentencas_agrupadas(texto, n=3)
                chunk_items = [{'text': t, 'meta': {}} for t in txts]
            else:
                txts = [c for c in est['splitter'].split_text(texto) if c.strip()]
                chunk_items = [{'text': t, 'meta': {}} for t in txts]
        except Exception as e:
            print(f'    ERRO ao dividir: {e}')
            continue

        chunk_items = [c for c in chunk_items if len(c['text'].strip()) >= 10]
        print(f'    {len(chunk_items)} chunks gerados')

        # Embeddings
        textos_chunks = [c['text'] for c in chunk_items]
        vetores = gerar_embeddings(textos_chunks)

        # Montar JSON
        dados = []
        for idx, (chunk, vetor) in enumerate(zip(chunk_items, vetores), 1):
            dados.append({
                'chunk_id':      f'{doc_id}_test{test_id:02d}_chunk{idx:04d}',
                'document_id':   doc_id,
                'document_name': doc_nome,
                'test_id':       test_id,
                'strategy':      est['strategy'],
                'chunk_size':    est['chunk_size'],
                'chunk_overlap': est['chunk_overlap'],
                'text':          chunk['text'],
                'embedding':     vetor,
                'metadata':      {'char_count': len(chunk['text']), **chunk['meta']},
            })

        # Salvar
        pasta_test = BASE_DIR / doc_id / f'test_{test_id:02d}'
        pasta_test.mkdir(parents=True, exist_ok=True)
        json_path = pasta_test / 'chunks_embeddings.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(dados, f, ensure_ascii=False, indent=2)

        sizes = [d['metadata']['char_count'] for d in dados]
        experimentos.append({
            'test_id':             test_id,
            'strategy':            est['strategy'],
            'chunk_size':          est['chunk_size'],
            'chunk_overlap':       est['chunk_overlap'],
            'num_chunks':          len(dados),
            'avg_chunk_size':      round(float(np.mean(sizes)), 1),
            'min_chunk_size':      min(sizes),
            'max_chunk_size':      max(sizes),
            'embedding_dimension': len(dados[0]['embedding']) if dados else 0,
        })

    summary_docs.append({'document_id': doc_id, 'document_name': doc_nome,
                         'total_chars': len(texto), 'experiments': experimentos})

# summary.json
summary_path = BASE_DIR / 'summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump({'documents': summary_docs}, f, ensure_ascii=False, indent=2)

print(f'\nPipeline concluído! Summary em: {summary_path}')

---
### 7. Download dos Resultados

In [ ]:
import shutil

# Compacta a pasta results/ para download
shutil.make_archive('/content/resultados_aula04', 'zip', '/content/results')
files.download('/content/resultados_aula04.zip')
print('Download iniciado!')